## 1. Konfigurasi

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import broadcast
import time

# ---- Parameter ----
HDFS = "hdfs://namenode:9000/data/oulad"
CUTOFF = None          #None = data seluruh kursus digunakan
REBUILD_JSON = False   #True = tulis ulang JSON dari CSV kalau walau sudah ada.

SUFFIX = "" if CUTOFF is None else f"_d{CUTOFF}"
KEYS = ["code_module", "code_presentation", "id_student"] #Data yang tidak masuk proses pelatihan

spark = (SparkSession.builder
         .appName("oulad-etl")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.hadoop.dfs.replication", "1")   #data node 1
         .getOrCreate())

def hdfs_path(p):
    jp = spark._jvm.org.apache.hadoop.fs.Path(p)
    return jp, jp.getFileSystem(spark._jsc.hadoopConfiguration())

def hdfs_exists(p):
    jp, fs = hdfs_path(p)
    return fs.exists(jp)

def hdfs_size_mb(p):
    jp, fs = hdfs_path(p)
    return fs.getContentSummary(jp).getLength() / (1024 * 1024)

print("Spark", spark.version, "| CUTOFF =", CUTOFF, "| suffix output =", repr(SUFFIX))

Spark 3.5.1 | CUTOFF = None | suffix output = ''


## 2. Extract

Nilai kosong di CSV OULAD ditulis sebagai ?. Dengan nullValue="?", Spark langsung menganggapnya null dan kolom seperti date_registration terbaca sebagai angka, bukan teks.

In [2]:
def read_csv(name):
    return (spark.read.option("header", True)
            .option("inferSchema", True)
            .option("nullValue", "?")
            .csv(f"{HDFS}/raw/csv/{name}.csv"))

student_info       = read_csv("studentInfo")
student_reg        = read_csv("studentRegistration")
student_assessment = read_csv("studentAssessment")
assessments        = read_csv("assessments")
vle                = read_csv("vle")
courses            = read_csv("courses")

for name, d in [("studentInfo", student_info), ("studentRegistration", student_reg),
                ("studentAssessment", student_assessment), ("assessments", assessments),
                ("vle", vle), ("courses", courses)]:
    print(f"{name:22s} {d.count():>8,} baris")
print("tipe date_registration:", dict(student_reg.dtypes)["date_registration"])

studentInfo              32,593 baris
studentRegistration      32,593 baris
studentAssessment       173,912 baris
assessments                 206 baris
vle                       6,364 baris
courses                      22 baris
tipe date_registration: int


### 2.2 Konversi studentVle.csv jadi JSON Lines di HDFS
Ini perubahan format saja, bukan transformasi data.

In [3]:
schema_vle_log = StructType([
    StructField("code_module", StringType()),
    StructField("code_presentation", StringType()),
    StructField("id_student", IntegerType()),
    StructField("id_site", IntegerType()),
    StructField("date", IntegerType()),
    StructField("sum_click", IntegerType()),
])

JSON_PATH = f"{HDFS}/raw/json/activityLog"

if REBUILD_JSON or not hdfs_exists(f"{JSON_PATH}/_SUCCESS"):
    t0 = time.time()
    (spark.read.option("header", True).schema(schema_vle_log)
          .csv(f"{HDFS}/raw/csv/studentVle.csv")
          .write.mode("overwrite").json(JSON_PATH))
    print(f"JSON ditulis dalam {time.time() - t0:.0f} detik")
else:
    print("JSON sudah ada di HDFS, penulisan dilewati")

log = spark.read.schema(schema_vle_log).json(JSON_PATH)
print(f"log aktivitas: {log.count():,} baris")

JSON sudah ada di HDFS, penulisan dilewati
log aktivitas: 10,655,280 baris


## 3. Transform

### 3.1 Validasi kualitas log aktivitas
Catatan: baris dengan kunci (siswa, situs, hari) yang sama **bukan duplikat yang dibuang**. Klik pada baris-baris itu dijumlahkan saat agregasi.

In [4]:
print("Jumlah null per kolom:")
log.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in log.columns]).show()

log.describe("date", "sum_click").show()

n_bad = log.filter(F.col("sum_click") <= 0).count()
print("sum_click <= 0:", n_bad)
assert n_bad == 0, "Ada sum_click tidak valid"

#date negatif = aktivitas sebelum kursus dimulai (wajar, tidak dibuang)
print("baris dengan date < 0:", log.filter(F.col("date") < 0).count())

Jumlah null per kolom:
+-----------+-----------------+----------+-------+----+---------+
|code_module|code_presentation|id_student|id_site|date|sum_click|
+-----------+-----------------+----------+-------+----+---------+
|          0|                0|         0|      0|   0|        0|
+-----------+-----------------+----------+-------+----+---------+

+-------+-----------------+------------------+
|summary|             date|         sum_click|
+-------+-----------------+------------------+
|  count|         10655280|          10655280|
|   mean|95.17399955702713|3.7169458709672574|
| stddev|76.07130084050435| 8.849046655101391|
|    min|              -25|                 1|
|    max|              269|              6977|
+-------+-----------------+------------------+

sum_click <= 0: 0
baris dengan date < 0: 688988


### 3.2 Fitur aktivitas VLE (1 baris per siswa-modul-presentasi)

In [5]:
log_f = log if CUTOFF is None else log.filter(F.col("date") <= CUTOFF)

#Ringkasan aktivitas
vle_agg = (log_f.groupBy(*KEYS).agg(
    F.sum("sum_click").alias("total_click"),
    F.countDistinct("date").alias("active_days"),
    F.countDistinct("id_site").alias("distinct_sites"),
    F.min("date").alias("first_activity_day"),
    F.max("date").alias("last_activity_day"),
).withColumn("avg_click_per_active_day",
             F.round(F.col("total_click") / F.col("active_days"), 3)))

#Klik per jenis aktivitas
vle_types = vle.select("id_site", "code_module", "code_presentation", "activity_type")
log_act = log_f.join(broadcast(vle_types), ["id_site", "code_module", "code_presentation"], "left")
n_no_type = log_act.filter(F.col("activity_type").isNull()).count()
print("baris tanpa activity_type:", n_no_type)
assert n_no_type == 0, "Ada id_site yang tidak ada di tabel vle"

act_agg = log_act.groupBy(*KEYS).pivot("activity_type").agg(F.sum("sum_click")).fillna(0)
for c in act_agg.columns:
    if c not in KEYS:
        act_agg = act_agg.withColumnRenamed(c, f"click_{c}")

#cache agar tidak dihitung ulang saat join dan saat Load
vle_features = vle_agg.join(act_agg, KEYS, "inner").cache()
print(f"vle_features: {vle_features.count():,} baris, {len(vle_features.columns)} kolom")

baris tanpa activity_type: 0
vle_features: 29,228 baris, 29 kolom


### 3.3 Fitur nilai tugas
Ujian akhir (Exam) dibuang karena skornya hampir identik dengan hasil akhir (data leakage).

In [6]:
assess = (student_assessment.join(assessments, "id_assessment", "inner")
          .filter(F.col("assessment_type") != "Exam"))

if CUTOFF is not None:
    assess = assess.filter(F.col("date_submitted") <= CUTOFF)

assess = (assess
    .withColumn("days_late", F.col("date_submitted") - F.col("date"))
    .withColumn("is_late", (F.col("days_late") > 0).cast("int")))

assess_features = (assess.groupBy(*KEYS).agg(
    F.count("*").alias("n_assessments"),
    F.round(F.avg("score"), 2).alias("avg_score"),
    F.min("score").alias("min_score"),
    F.max("score").alias("max_score"),
    F.sum("is_late").alias("n_late"),
    F.round(F.avg("days_late"), 2).alias("avg_days_late"),
    F.sum("is_banked").alias("n_banked"),
    F.round(F.avg(F.when(F.col("assessment_type") == "TMA", F.col("score"))), 2).alias("avg_score_tma"),
    F.round(F.avg(F.when(F.col("assessment_type") == "CMA", F.col("score"))), 2).alias("avg_score_cma"),
).cache())
print(f"assess_features: {assess_features.count():,} baris, {len(assess_features.columns)} kolom")

assess_features: 25,839 baris, 12 kolom


### 3.4 Penggabungan menjadi satu tabel fitur
studentInfo digunakan sebagai tabel dasar karena granularitas sudah sesuai (1 baris per siswa-modul-presentasi). Tabel hasil agregasi digabung dengan left join, sehingga siswa tanpa aktivitas atau tanpa tugas tetap ada.

- date_unregistration sengaja tidak dipakai karena hanya terisi untuk siswa yang mundur, sehingga membocorkan label.
- Penanda has_vle_activity dan has_assessment dibuat sebelum null diisi 0, agar informasi "tidak ada data" pada mahasiswa Withdrawn tidak hilang.
- Null pada kolom skor, imd_band, dan date_registration sengaja dibiarkan. Mengisinya adalah persiapan model dan dilakukan nanti, dipelajari dari data latih saja.

In [7]:
base = (student_info
        .join(courses, ["code_module", "code_presentation"], "left")
        .join(student_reg.select(*KEYS, "date_registration"), KEYS, "left"))

ml_df = (base.join(vle_features, KEYS, "left")
             .join(assess_features, KEYS, "left"))

ml_df = (ml_df
    .withColumn("has_vle_activity", F.col("total_click").isNotNull().cast("int"))
    .withColumn("has_assessment", F.col("n_assessments").isNotNull().cast("int")))

# Kolom jumlah/total: tidak ada aktivitas = 0
zero_cols = (["total_click", "active_days", "distinct_sites", "avg_click_per_active_day",
              "n_assessments", "n_late", "n_banked"]
             + [c for c in ml_df.columns if c.startswith("click_")])
ml_df = ml_df.fillna(0, subset=zero_cols)

# Label turunan: 1 = Pass/Distinction, 0 = Fail/Withdrawn
ml_df = ml_df.withColumn("label_binary",
    F.when(F.col("final_result").isin("Pass", "Distinction"), 1).otherwise(0))

print("jumlah kolom:", len(ml_df.columns))
ml_df.printSchema()

jumlah kolom: 52
root
 |-- code_module: string (nullable = true)
 |-- code_presentation: string (nullable = true)
 |-- id_student: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- region: string (nullable = true)
 |-- highest_education: string (nullable = true)
 |-- imd_band: string (nullable = true)
 |-- age_band: string (nullable = true)
 |-- num_of_prev_attempts: integer (nullable = true)
 |-- studied_credits: integer (nullable = true)
 |-- disability: string (nullable = true)
 |-- final_result: string (nullable = true)
 |-- module_presentation_length: integer (nullable = true)
 |-- date_registration: integer (nullable = true)
 |-- total_click: long (nullable = true)
 |-- active_days: long (nullable = true)
 |-- distinct_sites: long (nullable = true)
 |-- first_activity_day: integer (nullable = true)
 |-- last_activity_day: integer (nullable = true)
 |-- avg_click_per_active_day: double (nullable = false)
 |-- click_dataplus: long (nullable = true)
 |-- click_dua

## 4. Load
Hasil ditulis ke HDFS dalam format CSV agar dapat langsung digunakan untuk model machine learning.

In [8]:
OUT = f"{HDFS}/processed"

t0 = time.time()
(vle_features.coalesce(1).write.mode("overwrite")
    .option("header", True).csv(f"{OUT}/vle_features{SUFFIX}"))
(assess_features.coalesce(1).write.mode("overwrite")
    .option("header", True).csv(f"{OUT}/assess_features{SUFFIX}"))
(ml_df.coalesce(1).write.mode("overwrite")
    .option("header", True).csv(f"{OUT}/features_clean{SUFFIX}"))
print(f"Load selesai")

Load selesai


## 5. Verifikasi
Semua pemeriksaan dilakukan pada data yang dibaca kembali dari HDFS, bukan dari DataFrame di memori.

In [9]:
   final = (spark.read.option("header", True)
            .option("inferSchema", True)
            .csv(f"{OUT}/features_clean{SUFFIX}"))

n_final = final.count()
n_info  = student_info.count()
n_dupe  = final.groupBy(*KEYS).count().filter("count > 1").count()
str_cols = [c for c, t in final.dtypes if t == "string"]
n_qmark = sum(final.filter(F.col(c) == "?").count() for c in str_cols)

print(f"baris tabel fitur : {n_final:,} (studentInfo: {n_info:,})")
print(f"kunci ganda       : {n_dupe}")
print(f"nilai '?' tersisa : {n_qmark}")
print(f"tipe date_registration: {dict(final.dtypes)['date_registration']}")

assert n_final == n_info, "Jumlah baris berubah: kemungkinan join menggandakan atau menghilangkan baris"
assert n_dupe == 0, "Ada kunci ganda"
assert n_qmark == 0, "Masih ada '?' sebagai teks"
assert dict(final.dtypes)["date_registration"] == "int"
print("\nSemua pemeriksaan lulus.\n")

print("Sebaran target:")
final.groupBy("final_result").count().orderBy(F.desc("count")).show()
final.groupBy("label_binary").count().show()

print("Null per kolom (hanya yang > 0):")
nulls = final.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in final.columns]).first().asDict()
for c, v in nulls.items():
    if v > 0:
        print(f"  {c:25s} {v:,}")

baris tabel fitur : 32,593 (studentInfo: 32,593)
kunci ganda       : 0
nilai '?' tersisa : 0
tipe date_registration: int

Semua pemeriksaan lulus.

Sebaran target:
+------------+-----+
|final_result|count|
+------------+-----+
|        Pass|12361|
|   Withdrawn|10156|
|        Fail| 7052|
| Distinction| 3024|
+------------+-----+

+------------+-----+
|label_binary|count|
+------------+-----+
|           1|15385|
|           0|17208|
+------------+-----+

Null per kolom (hanya yang > 0):
  imd_band                  1,111
  date_registration         45
  first_activity_day        3,365
  last_activity_day         3,365
  avg_score                 6,777
  min_score                 6,777
  max_score                 6,777
  avg_days_late             6,754
  avg_score_tma             7,805
  avg_score_cma             17,493


## Selesai
Hasil ETL: `/data/oulad/processed/features_clean` (CSV, 1 baris per siswa-modul-presentasi), siap dipakai untuk tahap berikutnya.